In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
def llm(prompt):
    try:
        response = openai_client.responses.create(
            model="gpt-5.4-mini",
            input=prompt
        )
        return response.output_text
    except Exception as e:
        
        print(type(e).__name__)
        print(e)

NameError: name 'prompt' is not defined

In [4]:
context = '''
I just discovered the course. Can I still join?

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

# Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?

You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

# What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?

The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

# How should I start the course and follow the weekly workflow?

Start with the LLM Zoomcamp docs, the general Zoomcamp logistics docs, and the LLM Zoomcamp GitHub repository.

You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the course management platform.

A typical workflow is:

Watch the lesson videos.
Work through the lesson notebooks/code.
Read the homework instructions on GitHub.
Submit answers through the course platform before the deadline.
Homework is similar to the lesson flow, but uses a different dataset or slightly different task.


# Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?

When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

image #1

First field: This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.
Second field: Change this to your official name as in your identification documents—passport, national ID card, driver's license, etc. This is mandatory if you do not want "Lucid Elbakyan" on your certificate. This name will appear on your Certificate!
edit on GitHub
# Certificate: Can I follow the course in a self-paced mode and get a certificate?

No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.


# I missed the first homework - can I still get a certificate?

Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.


# Homework: Why does the content keep changing?

If the homework title contains [DRAFT], it means the homework is not ready yet.

The homework is ready only when both are true:

The homework form is open on the course management platform.
The homework title does not contain [DRAFT].
Until then, the content can still change. Working on the material or homework in advance is at your own risk, because the final version can be different.
'''

In [7]:
question = 'I just discovered the course. Can i join now?'

In [8]:
prompt = f'''
your task is to answer questions from the course participants based on the 
provided context.

Use the context to find relavent information and provide accurate answers.

Questions:
{question}

Context:
{context}
'''

In [9]:
print(prompt)


your task is to answer questions from the course participants based on the 
provided context.

Use the context to find relavent information and provide accurate answers.

Questions:
I just discovered the course. Can i join now?

Context:

I just discovered the course. Can I still join?

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

# Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?

You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

# What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?

The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is p

In [11]:

answer = llm(prompt)
print(answer)

Yes, you can still join now.

You can start learning anytime, and if you want a certificate, make sure to submit your project while submissions are still open.


In [13]:
#make rag 

def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [14]:
import requests


docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [15]:
# fetch all the FAQ documents from all courses:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1368

In [16]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [17]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [18]:
search_results = index.search(
                question, 
                boost_dict={'question': 2.0, 'section':0.5},
                filter_dict={'course':'llm-zoomcamp'},
                num_results = 5)

In [19]:
#rag engine of search (retrival step)

def search(question, course='llm-zoomcamp'):
    boost_dict = {
        'question': 2.0,
        'section': 0.5
    }

    filter_dict = {
        'course': course
    }

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [20]:
search_results = search(question)

In [21]:
#step 2 prompt (split into 2 parts)(instruction)

INSTRUCTIONS = '''
your task is to answer questions from the course participants based on the 
provided context.

Use the context to find relavent information and provide accurate answers. if the answer is not found in the context,
respond with "i don't know."
'''

In [22]:
#step 2(Prompt)(user prompt)

USER_PROMPT_TEMPLATE = f'''

Questions: "What are the courses that i can join?"

Context:
{context}
'''

In [23]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [24]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [25]:
def build_prompt(question, search_results):
    context = build_context (search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question = question, 
        context= context
    )
    return prompt.strip()

In [26]:
prompt = build_prompt(question, search_results)

In [27]:
print(prompt)

Questions: "What are the courses that i can join?"

Context:

I just discovered the course. Can I still join?

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

# Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?

You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

# What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?

The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram and Slack before it begins. You can also watch live on the DataTalksClub You

In [28]:
from openai import OpenAI
openai_client = OpenAI()

In [29]:
response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=prompt
    )

In [30]:
response.output_text

'Here are the answers to your questions (based on the course info you shared):\n\n### 1) What courses can I join?\nYou can join the course(s) mentioned in the program announcements/pages. If you mean the specific course you just discovered, **yes—you can still join**, but **to receive a certificate you must submit your project while submissions are still open**.\n\n### 2) I just discovered the course—can I still join?\n**Yes**, you can join.  \nHowever, **certificate eligibility depends on deadlines**: you need to submit your project while the course is still accepting submissions.\n\n### 3) LLM Zoomcamp: When will I get the confirmation email?\nYou **don’t need a confirmation email**.  \n- You’re accepted already.\n- You can start learning and submitting homework **while the form is open**.\n- Registration is mainly for gauging interest before the start date.\n\n### 4) Where is the Zoom link for Office Hours / live sessions?\nThe Zoom link is **only published to instructors/presenters

In [31]:
response.output[0].content[0].text

'Here are the answers to your questions (based on the course info you shared):\n\n### 1) What courses can I join?\nYou can join the course(s) mentioned in the program announcements/pages. If you mean the specific course you just discovered, **yes—you can still join**, but **to receive a certificate you must submit your project while submissions are still open**.\n\n### 2) I just discovered the course—can I still join?\n**Yes**, you can join.  \nHowever, **certificate eligibility depends on deadlines**: you need to submit your project while the course is still accepting submissions.\n\n### 3) LLM Zoomcamp: When will I get the confirmation email?\nYou **don’t need a confirmation email**.  \n- You’re accepted already.\n- You can start learning and submitting homework **while the form is open**.\n- Registration is mainly for gauging interest before the start date.\n\n### 4) Where is the Zoom link for Office Hours / live sessions?\nThe Zoom link is **only published to instructors/presenters

In [32]:
response.usage

ResponseUsage(input_tokens=788, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=751, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1539)

In [33]:
message_history = [
    {'role': 'developer' , 'content' : INSTRUCTIONS},
    {'role': 'user', 'content' : prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input= message_history
)

In [34]:
def llm(instructions, user_prompt, model="gpt-5.4-nano"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [35]:
#llm model
def rag(query, model="gpt-5.4-nano"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [36]:
answer = rag('questions')
print(answer)

Based on the provided context, I can’t see a list of the specific courses you can join. I only see details about **LLM Zoomcamp** and information about certificates/self-paced vs live cohorts.

So: **i don't know** what the courses are that you can join.
